# Regression Task: Wind Interpolation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import torch
import gpytorch
from tqdm.auto import tqdm
from linear_operator import settings
from gpytorch import settings as gsettings
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP
from grf_gp.utils.config import set_gp_defaults

project_root = os.path.abspath('../../..')
sys.path.append(project_root)
sys.path.append(os.path.abspath('.'))
from data_utils import prepare_wind_graph_data
from experiments.utils import train_model, evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_gp_defaults(settings, gsettings)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
NC_FILE = os.path.join(project_root, 'data/wind_interpolation/aac1241c2ceff963a460829f1c68b132.nc')
USE_DOWNSAMPLING = True
DOWNSAMPLE_FACTOR = 10
MAX_WALK_LENGTH = 5
WALKS_PER_NODE = 10**4
P_HALT = 0.05
LR = 0.01
MAX_ITER = 300
SEED = 0

In [10]:
np.random.seed(SEED)
torch.manual_seed(SEED)

data = prepare_wind_graph_data(
    NC_FILE,
    use_downsampling=USE_DOWNSAMPLING,
    downsample_factor=DOWNSAMPLE_FACTOR,
    dtype=np.float64,
)

X_train = torch.tensor(data['X_train']).long().to(device)
Y_train = torch.tensor(data['y_train'], dtype=torch.float64).flatten().to(device)
X_test = torch.tensor(data['X_test']).long().to(device)
Y_test = torch.tensor(data['y_test'], dtype=torch.float64).flatten().to(device)
X_full = torch.tensor(data['X']).long().to(device)

adjacency_matrix = data['adjacency'].toarray()
L = get_normalized_laplacian(adjacency_matrix)
L = torch.tensor(L, dtype=torch.float64).to(device)
orig_std = float(data['y_std'])

## Random Walk Sampling

In [11]:
sampler = GRFSampler(L, WALKS_PER_NODE, P_HALT, MAX_WALK_LENGTH, seed=SEED, use_tqdm=True, n_processes=4)
rw_mats = sampler()

## General GRF

In [12]:
likelihood_general = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_general = GeneralGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model_general = GRFGP(X_train, Y_train, likelihood_general, kernel_general).to(device)
_ = train_model(model_general, likelihood_general, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_g, rmse_g, nlpd_g = evaluate_model(model_general, likelihood_general, X_train, Y_train, X_test, Y_test, orig_std)
print(f'GeneralGRFKernel | LML: {lml_g:.4f} | RMSE: {rmse_g:.4f} | NLPD: {nlpd_g:.4f}')
print('modulation:', kernel_general.modulation_function.detach().cpu().numpy())

Training:   0%|          | 0/300 [00:00<?, ?it/s]

GeneralGRFKernel | LML: -341.3529 | RMSE: 2.2377 | NLPD: 1.1961
modulation: [ 1.9356366  -0.98609704 -1.8660166   1.0700904  -0.04240725]


## Diffusion GRF

In [13]:
likelihood_diff = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_diff = DiffusionGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model_diff = GRFGP(X_train, Y_train, likelihood_diff, kernel_diff).to(device)
_ = train_model(model_diff, likelihood_diff, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_d, rmse_d, nlpd_d = evaluate_model(model_diff, likelihood_diff, X_train, Y_train, X_test, Y_test, orig_std)
print(f'DiffusionGRFKernel | LML: {lml_d:.4f} | RMSE: {rmse_d:.4f} | NLPD: {nlpd_d:.4f}')
print(f'beta: {kernel_diff.beta.item():.4f}, sigma_f: {kernel_diff.sigma_f.item():.4f}')

Training:   0%|          | 0/300 [00:00<?, ?it/s]

DiffusionGRFKernel | LML: -205.6600 | RMSE: 2.3396 | NLPD: 1.2552
beta: 2.2446, sigma_f: 1.8095
